In [1]:
import pandas as pd

df=pd.read_csv(r'D:\E Drive\Sentiflow-Network Intrusion Detection\Data\Consolidated_df.csv')
df

,duration,protocoltype,service,flag,srcbytes,dstbytes,land,wrongfragment,urgent,hot,...,serror_rate_gap,rerror_rate_gap,authentication_risk_score,has_failed_login,failed_login_and_logged_in,suspicious_admin_activity,privileged_activity_score,content_risk_score,file_and_shell_activity,root_compromise_ratio
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.00,0.05,0,0,0,0,0,0,0,0.0
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.17,0.00,0,0,0,0,0,0,0,0.0
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.72,0.01,0,0,0,0,0,0,0,0.0
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.00,0.00,0,0,0,0,0,0,0,0.0


In [2]:
df.columns

Index(['duration', 'protocoltype', 'service', 'flag', 'srcbytes', 'dstbytes',
       'land', 'wrongfragment', 'urgent', 'hot', 'numfailedlogins', 'loggedin',
       'numcompromised', 'rootshell', 'suattempted', 'numroot',
       'numfilecreations', 'numshells', 'numaccessfiles', 'numoutboundcmds',
       'ishostlogin', 'isguestlogin', 'count', 'srvcount', 'serrorrate',
       'srvserrorrate', 'rerrorrate', 'srvrerrorrate', 'samesrvrate',
       'diffsrvrate', 'srvdiffhostrate', 'dsthostcount', 'dsthostsrvcount',
       'dsthostsamesrvrate', 'dsthostdiffsrvrate', 'dsthostsamesrcportrate',
       'dsthostsrvdiffhostrate', 'dsthostserrorrate', 'dsthostsrvserrorrate',
       'dsthostrerrorrate', 'dsthostsrvrerrorrate', 'attack', 'lastflag',
       'attack_category', 'binary_target', 'total_bytes', 'bytes_per_second',
       'src_dst_byte_ratio', 'src_byte_fraction', 'dst_byte_fraction',
       'byte_asymmetry', 'service_connection_ratio', 'host_service_ratio',
       'different_service_con

In [3]:
print(df['binary_target'].value_counts())
print(df['attack_category'].value_counts())

binary_target
Normal    67343
Attack    58630
Name: count, dtype: int64
attack_category
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["binary_target"])
y = df["binary_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42)

In [5]:
ORIGINAL_FEATURES = (
    "duration",
    "protocoltype",
    "service",
    "flag",
    "srcbytes",
    "dstbytes",
    "land",
    "wrongfragment",
    "urgent",
    "hot",
    "numfailedlogins",
    "loggedin",
    "numcompromised",
    "rootshell",
    "suattempted",
    "numroot",
    "numfilecreations",
    "numshells",
    "numaccessfiles",
    "numoutboundcmds",
    "ishostlogin",
    "isguestlogin",
    "count",
    "srvcount",
    "serrorrate",
    "srvserrorrate",
    "rerrorrate",
    "srvrerrorrate",
    "samesrvrate",
    "diffsrvrate",
    "srvdiffhostrate",
    "dsthostcount",
    "dsthostsrvcount",
    "dsthostsamesrvrate",
    "dsthostdiffsrvrate",
    "dsthostsamesrcportrate",
    "dsthostsrvdiffhostrate",
    "dsthostserrorrate",
    "dsthostsrvserrorrate",
    "dsthostrerrorrate",
    "dsthostsrvrerrorrate",
)


ENGINEERED_GROUPS = {
    "traffic_volume": (
        "total_bytes",
        "bytes_per_second",
        "src_dst_byte_ratio",
        "src_byte_fraction",
        "dst_byte_fraction",
        "byte_asymmetry",
    ),

    "connection_concentration": (
        "service_connection_ratio",
        "host_service_ratio",
        'different_service_connections',
        'different_host_service_connections'
    ),

    "scanning": (
        "short_term_scan_pressure",
        "host_scan_pressure",
        "same_source_port_pressure",
    ),

    "errors": (
        "short_term_serror_score",
        "short_term_rerror_score",
        "short_term_error_score",
        "host_serror_score",
        "host_rerror_score",
        "host_error_score",
    ),

    "behavior_gaps": (
        "same_service_rate_gap",
        "different_service_rate_gap",
        "serror_rate_gap",
        "rerror_rate_gap",
    ),

    "authentication": (
        "authentication_risk_score",
        "has_failed_login",
        "failed_login_and_logged_in",
        "suspicious_admin_activity",
    ),

    "privilege_and_content": (
        "privileged_activity_score",
        "content_risk_score",
        "file_and_shell_activity",
        "root_compromise_ratio",
    ),

}

ENGINEERED_FEATURES = tuple(
    feature
    for group in ENGINEERED_GROUPS.values()
    for feature in group
)

COMBINED_FEATURES = (
    ORIGINAL_FEATURES
    + ENGINEERED_FEATURES
)

FEATURE_SETS = {
    "original": ORIGINAL_FEATURES,
    "engineered": ENGINEERED_FEATURES,
    "combined": COMBINED_FEATURES
}

In [6]:
X_train_original = X_train[list(ORIGINAL_FEATURES)]
X_test_original = X_test[list(ORIGINAL_FEATURES)]
X_train_engineered = X_train[list(ENGINEERED_FEATURES)]
X_test_engineered = X_test[list(ENGINEERED_FEATURES)]
X_train_combined = X_train[list(COMBINED_FEATURES)]
X_test_combined = X_test[list(COMBINED_FEATURES)]

In [7]:
X_train_original.columns

Index(['duration', 'protocoltype', 'service', 'flag', 'srcbytes', 'dstbytes',
       'land', 'wrongfragment', 'urgent', 'hot', 'numfailedlogins', 'loggedin',
       'numcompromised', 'rootshell', 'suattempted', 'numroot',
       'numfilecreations', 'numshells', 'numaccessfiles', 'numoutboundcmds',
       'ishostlogin', 'isguestlogin', 'count', 'srvcount', 'serrorrate',
       'srvserrorrate', 'rerrorrate', 'srvrerrorrate', 'samesrvrate',
       'diffsrvrate', 'srvdiffhostrate', 'dsthostcount', 'dsthostsrvcount',
       'dsthostsamesrvrate', 'dsthostdiffsrvrate', 'dsthostsamesrcportrate',
       'dsthostsrvdiffhostrate', 'dsthostserrorrate', 'dsthostsrvserrorrate',
       'dsthostrerrorrate', 'dsthostsrvrerrorrate'],
      dtype='object')

## Original features: baseline models tracked with MLflow

The preprocessing pipeline is fitted independently inside each model run. Numeric missing values are median-imputed and standardized; categorical missing values are most-frequent-imputed and one-hot encoded. This prevents information from the test set leaking into preprocessing. The positive class is `Attack`.

In [20]:
import matplotlib
matplotlib.use("Agg")  # Allows plot artifact creation in headless environments.
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import ngboost
import numpy as np
import sklearn
import xgboost
from mlflow.models import infer_signature
from ngboost import NGBClassifier
from sklearn.base import ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Insert project root at the beginning
sys.path.insert(0, str(PROJECT_ROOT))

#print(sys.path[0])
from configs.config import TRACKING_URI, EXPERIMENT_NAME, RANDOM_STATE, N_SPLITS
# Use one absolute database path whether Jupyter starts in the project root or Notebooks/.
TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'Notebooks' / 'mlflow.db').resolve().as_posix()}"

d:\E Drive\Sentiflow-Network Intrusion Detection


In [24]:
import json
import platform
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # Allows plot artifact creation in headless environments.
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import sklearn
import xgboost
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
#from configs.config import TRACKING_URI, EXPERIMENT_NAME, RANDOM_STATE


mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

categorical_features = X_train_original.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numeric_features = [
    column for column in X_train_original.columns
    if column not in categorical_features
]

# Make Attack the positive class so precision/recall/F1 have the intended meaning.
label_mapping = {"Normal": 0, "Attack": 1}
y_train_encoded = y_train.astype(str).map(label_mapping)
y_test_encoded = y_test.astype(str).map(label_mapping)
if y_train_encoded.isna().any() or y_test_encoded.isna().any():
    unknown_labels = sorted(
        set(y_train.astype(str)).union(y_test.astype(str)) - set(label_mapping)
    )
    raise ValueError(f"Unexpected target labels: {unknown_labels}")

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # Dense output is required by GaussianNB.
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

class CompatibleNGBClassifier(ClassifierMixin, NGBClassifier):
    """Expose classifier metadata required by scikit-learn probability scorers."""
    def fit(self, X, y, **fit_params):
        fitted = super().fit(X, y, **fit_params)
        self.classes_ = np.unique(y)
        return fitted

models = {
    "logisticreg": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "naivebayes": GaussianNB(),
    "ngboost": CompatibleNGBClassifier(
        n_estimators=200, learning_rate=0.05,
        random_state=RANDOM_STATE, verbose=False
    ),
    "decisiontree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    "randomforest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "xgboost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

def binary_metrics(y_true, y_pred, y_score, split):
    return {
        f"{split}_accuracy": accuracy_score(y_true, y_pred),
        f"{split}_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{split}_precision": precision_score(y_true, y_pred, zero_division=0),
        f"{split}_recall": recall_score(y_true, y_pred, zero_division=0),
        f"{split}_f1": f1_score(y_true, y_pred, zero_division=0),
        f"{split}_roc_auc": roc_auc_score(y_true, y_score),
        f"{split}_mcc": matthews_corrcoef(y_true, y_pred),
    }

data_path = Path("Data/Consolidated_df.csv")
if not data_path.exists():
    data_path = Path("../Data/Consolidated_df.csv")
data_source = str(data_path.resolve())

mlflow_train_df = X_train_original.copy()
mlflow_train_df["binary_target"] = y_train
mlflow_test_df = X_test_original.copy()
mlflow_test_df["binary_target"] = y_test
train_dataset = mlflow.data.from_pandas(
    mlflow_train_df, source=data_source, targets="binary_target",
    name="original_nfs_train"
)
test_dataset = mlflow.data.from_pandas(
    mlflow_test_df, source=data_source, targets="binary_target",
    name="original_nfs_test"
)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Categorical features: {categorical_features}")
print(f"Numeric features: {len(numeric_features)}")

d:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Tracking URI: sqlite:///mlflow.db
Experiment: Network_Intrusion_Detection
Categorical features: ['protocoltype', 'service', 'flag']
Numeric features: 38


d:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


In [25]:
results = []

for algorithm_name, estimator in models.items():
    run_name = f"Original_NFS_{algorithm_name}"
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", estimator),
    ])

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "data_variant": "Original_NFS",
            "feature_set": "original",
            "task": "binary_classification",
            "positive_class": "Attack",
            "algorithm": algorithm_name,
        })
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(test_dataset, context="evaluation")
        mlflow.log_params({
            "algorithm": algorithm_name,
            "random_state": RANDOM_STATE,
            "raw_feature_count": X_train_original.shape[1],
            "numeric_feature_count": len(numeric_features),
            "categorical_feature_count": len(categorical_features),
            "train_row_count": len(X_train_original),
            "test_row_count": len(X_test_original),
            "numeric_imputation": "median",
            "numeric_scaling": "StandardScaler",
            "categorical_imputation": "most_frequent",
            "categorical_encoding": "OneHotEncoder(handle_unknown=ignore)",
        })
        model_params = {
            f"model_{key}": value
            for key, value in estimator.get_params(deep=False).items()
            if value is not None
        }
        mlflow.log_params(model_params)

        fit_started = time.perf_counter()
        pipeline.fit(X_train_original, y_train_encoded)
        fit_seconds = time.perf_counter() - fit_started

        inference_started = time.perf_counter()
        test_pred = pipeline.predict(X_test_original)
        test_score = pipeline.predict_proba(X_test_original)[:, 1]
        inference_seconds = time.perf_counter() - inference_started
        train_pred = pipeline.predict(X_train_original)
        train_score = pipeline.predict_proba(X_train_original)[:, 1]

        metrics = {
            **binary_metrics(y_train_encoded, train_pred, train_score, "train"),
            **binary_metrics(y_test_encoded, test_pred, test_score, "test"),
            "fit_seconds": fit_seconds,
            "test_inference_seconds": inference_seconds,
        }
        tn, fp, fn, tp = confusion_matrix(
            y_test_encoded, test_pred, labels=[0, 1]
        ).ravel()
        metrics.update({
            "test_true_negatives": int(tn),
            "test_false_positives": int(fp),
            "test_false_negatives": int(fn),
            "test_true_positives": int(tp),
            "test_specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "test_false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        })
        mlflow.log_metrics(metrics)

        report = classification_report(
            y_test_encoded,
            test_pred,
            labels=[0, 1],
            target_names=["Normal", "Attack"],
            output_dict=True,
            zero_division=0,
        )
        mlflow.log_dict(report, "metrics/test_classification_report.json")
        mlflow.log_dict({
            "label_mapping": label_mapping,
            "original_features": list(X_train_original.columns),
            "categorical_features": categorical_features,
            "numeric_features": numeric_features,
            "class_distribution_train": y_train.value_counts().to_dict(),
            "class_distribution_test": y_test.value_counts().to_dict(),
            "library_versions": {
                "python": platform.python_version(),
                "scikit_learn": sklearn.__version__,
                "mlflow": mlflow.__version__,
                "ngboost": ngboost.__version__,
                "xgboost": xgboost.__version__,
            },
        }, "metadata/run_metadata.json")

        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test_encoded, test_pred,
            display_labels=["Normal", "Attack"],
            cmap="Blues", ax=ax, colorbar=False
        )
        ax.set_title(f"{run_name} - test confusion matrix")
        fig.tight_layout()
        mlflow.log_figure(fig, "plots/test_confusion_matrix.png")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(5, 4))
        RocCurveDisplay.from_predictions(
            y_test_encoded, test_score, name=algorithm_name, ax=ax
        )
        ax.set_title(f"{run_name} - test ROC curve")
        fig.tight_layout()
        mlflow.log_figure(fig, "plots/test_roc_curve.png")
        plt.close(fig)

        input_example = X_train_original.head(5).copy()
        # Float schema permits numeric missing values at future inference time.
        input_example[numeric_features] = input_example[numeric_features].astype("float64")
        signature = infer_signature(
            input_example, pipeline.predict(input_example)
        )
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            signature=signature,
            input_example=input_example,
        )

        results.append({
            "run_name": run_name,
            "run_id": run.info.run_id,
            **metrics,
        })
        print(f"Completed {run_name}: test F1={metrics['test_f1']:.4f}")

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

d:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/03 16:38:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can

Completed Original_NFS_logisticreg: test F1=0.9727


2026/08/03 16:38:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed Original_NFS_naivebayes: test F1=0.8298


2026/08/03 16:38:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/08/03 16:38:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed Original_NFS_randomforest: test F1=0.9991


2026/08/03 16:38:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed Original_NFS_xgboost: test F1=0.9993


,run_name,run_id,train_accuracy,train_balanced_accuracy,train_precision,train_recall,train_f1,train_roc_auc,train_mcc,test_accuracy,...,test_roc_auc,test_mcc,fit_seconds,test_inference_seconds,test_true_negatives,test_false_positives,test_false_negatives,test_true_positives,test_specificity,test_false_positive_rate
0,Original_NFS_xgboost,c84d8391070d414aaec024a2a0873196,0.999851,0.999851,0.999829,0.999851,0.999840,1.000000,0.999701,0.999325,...,0.999994,0.998644,1.680178,0.105219,13463,6,11,11715,0.999555,0.000445
1,Original_NFS_randomforest,7395008fafff4bb9b65da091574357da,0.999960,0.999961,0.999936,0.999979,0.999957,1.000000,0.999920,0.999127,...,0.999996,0.998245,2.968053,0.215001,13463,6,16,11710,0.999555,0.000445
2,Original_NFS_decisiontree,11747ddcc3df4276a24288e3aeb168ef,0.999960,0.999957,1.000000,0.999915,0.999957,1.000000,0.999920,0.998095,...,0.998150,0.996172,1.684193,0.080161,13441,28,20,11706,0.997921,0.002079
3,Original_NFS_logisticreg,74448f61e6944cd6ada454e6fbcbe86d,0.972762,0.972170,0.977549,0.963607,0.970528,0.996435,0.945296,0.974678,...,0.996700,0.949120,2.259125,0.075528,13208,261,377,11349,0.980622,0.019378
4,Original_NFS_naivebayes,24ab4f262dee43989065c8a37ff1d11c,0.858650,0.848307,0.996473,0.698768,0.821480,0.980639,0.741063,0.864100,...,0.981478,0.749914,0.554268,0.197922,13427,42,3382,8344,0.996882,0.003118


## Five-fold cross-validation versus held-out test performance

Cross-validation is performed only on `X_train_original` and `y_train_encoded`. The complete preprocessing and model pipeline is cloned and refitted inside every fold, preventing leakage from validation folds into imputation, scaling, or encoding. The untouched held-out test metrics are shown beside the CV mean and standard deviation.

In [ ]:
cv_splitter = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
cv_scorers = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
cv_summary_rows = []
cv_fold_rows = []
test_results_by_run = results_df.set_index("run_name")

for algorithm_name, estimator in models.items():
    run_name = f"Original_NFS_{algorithm_name}"
    cv_pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", clone(estimator)),
    ])
    cv_output = cross_validate(
        cv_pipeline, X_train_original, y_train_encoded,
        cv=cv_splitter, scoring=cv_scorers,
        return_train_score=False, n_jobs=1, error_score="raise",
    )

    test_row = test_results_by_run.loc[run_name]
    summary = {"model": algorithm_name}
    mlflow_cv_metrics = {}
    algorithm_fold_rows = []
    for metric_name in cv_scorers:
        fold_values = cv_output[f"test_{metric_name}"]
        cv_mean = float(np.mean(fold_values))
        cv_std = float(np.std(fold_values, ddof=1))
        test_value = float(test_row[f"test_{metric_name}"])
        summary.update({
            f"cv_{metric_name}_mean": cv_mean,
            f"cv_{metric_name}_std": cv_std,
            f"test_{metric_name}": test_value,
            f"test_minus_cv_{metric_name}": test_value - cv_mean,
        })
        mlflow_cv_metrics.update({
            f"cv_{metric_name}_mean": cv_mean,
            f"cv_{metric_name}_std": cv_std,
            f"test_minus_cv_{metric_name}": test_value - cv_mean,
        })
        for fold_number, fold_value in enumerate(fold_values, start=1):
            mlflow_cv_metrics[f"cv_{metric_name}_fold_{fold_number}"] = float(fold_value)

    for fold_index in range(N_SPLITS):
        fold_row = {
            "model": algorithm_name, "fold": fold_index + 1,
            "fit_seconds": float(cv_output["fit_time"][fold_index]),
            "score_seconds": float(cv_output["score_time"][fold_index]),
        }
        fold_row.update({
            metric: float(cv_output[f"test_{metric}"][fold_index])
            for metric in cv_scorers
        })
        algorithm_fold_rows.append(fold_row)
        cv_fold_rows.append(fold_row)

    summary["cv_fit_seconds_mean"] = float(np.mean(cv_output["fit_time"]))
    cv_summary_rows.append(summary)
    with mlflow.start_run(run_id=str(test_row["run_id"])):
        mlflow.set_tags({
            "cross_validation": "StratifiedKFold",
            "cv_n_splits": str(N_SPLITS),
            "cv_preprocessing_inside_fold": "true",
        })
        mlflow.log_metrics(mlflow_cv_metrics)
        mlflow.log_dict(
            {"folds": algorithm_fold_rows},
            "metrics/cross_validation_folds.json",
        )
    print(
        f"{algorithm_name}: CV F1={summary['cv_f1_mean']:.4f} +/- "
        f"{summary['cv_f1_std']:.4f}; test F1={summary['test_f1']:.4f}"
    )

cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values(
    "cv_f1_mean", ascending=False
).reset_index(drop=True)
cv_fold_results_df = pd.DataFrame(cv_fold_rows)
comparison_columns = [
    "model", "cv_accuracy_mean", "cv_accuracy_std", "test_accuracy",
    "cv_precision_mean", "cv_precision_std", "test_precision",
    "cv_recall_mean", "cv_recall_std", "test_recall",
    "cv_f1_mean", "cv_f1_std", "test_f1", "test_minus_cv_f1",
    "cv_roc_auc_mean", "cv_roc_auc_std", "test_roc_auc",
]
cv_vs_test_df = cv_summary_df[comparison_columns]
cv_vs_test_df.style.format({
    column: "{:.6f}" for column in cv_vs_test_df.columns if column != "model"
})

## Best model after cross-validation: XGBoost

Based on both five-fold stratified cross-validation and the held-out test set, **XGBoost is the best overall model** for the original NFS features.

| Model | CV F1 (mean +/- SD) | Test F1 | Test accuracy | Test recall | Test ROC-AUC | FP | FN |
|---|---:|---:|---:|---:|---:|---:|---:|
| **XGBoost** | **99.8891% +/- 0.0103%** | **99.9275%** | **99.9325%** | **99.9062%** | 99.9994% | **6** | **11** |
| Random Forest | 99.8624% +/- 0.0243% | 99.9062% | 99.9127% | 99.8636% | **99.9996%** | **6** | 16 |
| Decision Tree | 99.7975% +/- 0.0294% | 99.7954% | 99.8095% | 99.8294% | 99.8150% | 28 | 20 |
| NGBoost | 98.6039% +/- 0.0975% | 98.8135% | 98.9006% | 98.3626% | 99.9530% | 85 | 192 |
| Logistic Regression | 96.9980% +/- 0.1735% | 97.2660% | 97.4678% | 96.7849% | 99.6700% | 261 | 377 |
| Gaussian Naive Bayes | 82.2230% +/- 0.4202% | 82.9753% | 86.4100% | 71.1581% | 98.1478% | 42 | 3,382 |

XGBoost has the highest mean CV F1 and test F1. It missed only 11 attacks, compared with 16 for Random Forest, while producing the same six false positives. Its test-minus-CV F1 gap is only about 0.038 percentage points, indicating stable performance across the folds and held-out test split. Random Forest is a very close second and has a marginally higher ROC-AUC. NGBoost performs well and provides probabilistic modeling, but ranks fourth here and requires substantially more training time.

> **Selection:** Use **XGBoost** as the current best model. Because the leading scores are near-perfect, also validate with an independent or temporal test set and check for duplicate records or target leakage before production deployment.

### MLflow location

The completed test and cross-validation metrics for all six runs are stored in `Notebooks/mlflow.db` under the `Network_Intrusion_Detection` experiment. Launch the UI from the project root with:

```powershell
.\.venv\Scripts\mlflow.exe ui --backend-store-uri "sqlite:///Notebooks/mlflow.db"
```

# Original-feature multiclass attack classification

This section extends the same six algorithms to predict `attack_category`: **Normal, DoS, Probe, R2L, and U2R**. It recreates the original train/test split and uses only the 41 original traffic features. Balanced training weights reduce domination by Normal and DoS traffic. Run names follow `original_Multiclass_{Algorithm}`.

In [1]:
import json, platform, sys, time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
import ngboost, numpy as np, pandas as pd, sklearn, xgboost
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from ngboost import NGBClassifier
from ngboost.distns import k_categorical
from sklearn.base import ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, matthews_corrcoef, precision_recall_fscore_support, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
PROJECT_ROOT_MC=Path.cwd()
if not (PROJECT_ROOT_MC/'configs').exists(): PROJECT_ROOT_MC=PROJECT_ROOT_MC.parent
sys.path.insert(0,str(PROJECT_ROOT_MC))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE
mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_ROOT_MC/'Notebooks'/'mlflow.db').resolve().as_posix()}"); mlflow.set_experiment(EXPERIMENT_NAME)
data_path_mc=PROJECT_ROOT_MC/'Data'/'Consolidated_df.csv'; multiclass_df=pd.read_csv(data_path_mc)
ORIGINAL_FEATURES_MC=('duration','protocoltype','service','flag','srcbytes','dstbytes','land','wrongfragment','urgent','hot','numfailedlogins','loggedin','numcompromised','rootshell','suattempted','numroot','numfilecreations','numshells','numaccessfiles','numoutboundcmds','ishostlogin','isguestlogin','count','srvcount','serrorrate','srvserrorrate','rerrorrate','srvrerrorrate','samesrvrate','diffsrvrate','srvdiffhostrate','dsthostcount','dsthostsrvcount','dsthostsamesrvrate','dsthostdiffsrvrate','dsthostsamesrcportrate','dsthostsrvdiffhostrate','dsthostserrorrate','dsthostsrvserrorrate','dsthostrerrorrate','dsthostsrvrerrorrate')
train_mc_idx,test_mc_idx=train_test_split(np.arange(len(multiclass_df)),test_size=0.20,stratify=multiclass_df['binary_target'],random_state=RANDOM_STATE)
X_train_mc=multiclass_df.iloc[train_mc_idx][list(ORIGINAL_FEATURES_MC)].copy(); X_test_mc=multiclass_df.iloc[test_mc_idx][list(ORIGINAL_FEATURES_MC)].copy()
MULTICLASS_CLASSES=['Normal','DoS','Probe','R2L','U2R']; multiclass_mapping={name:i for i,name in enumerate(MULTICLASS_CLASSES)}; inverse_multiclass_mapping={i:name for name,i in multiclass_mapping.items()}
y_train_mc_text=multiclass_df.iloc[train_mc_idx]['attack_category'].astype(str); y_test_mc_text=multiclass_df.iloc[test_mc_idx]['attack_category'].astype(str); y_train_mc=y_train_mc_text.map(multiclass_mapping).to_numpy(); y_test_mc=y_test_mc_text.map(multiclass_mapping).to_numpy()
categorical_mc=X_train_mc.select_dtypes(include=['object','category','string']).columns.tolist(); numeric_mc=[c for c in ORIGINAL_FEATURES_MC if c not in categorical_mc]
balanced_train_weights=compute_sample_weight(class_weight='balanced',y=y_train_mc)
print('Classes:',multiclass_mapping); display(pd.DataFrame({'train':y_train_mc_text.value_counts(),'test':y_test_mc_text.value_counts()}).fillna(0).astype(int))

Classes: {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}


,train,test
attack_category,,
Normal,53874,13469
DoS,36668,9259
Probe,9367,2289
R2L,825,170
U2R,44,8


In [2]:
def make_multiclass_preprocessor(): return ColumnTransformer([('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),numeric_mc),('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical_mc)],remainder='drop',verbose_feature_names_out=False)
class CompatibleMultiNGBClassifier(ClassifierMixin,NGBClassifier):
    def fit(self,X,y,**fit_params): fitted=super().fit(X,y,**fit_params); self.classes_=np.arange(len(MULTICLASS_CLASSES)); return fitted
multiclass_models={
 'logisticreg':LogisticRegression(max_iter=1500,random_state=RANDOM_STATE),
 'naivebayes':GaussianNB(),
 'ngboost':CompatibleMultiNGBClassifier(Dist=k_categorical(len(MULTICLASS_CLASSES)),n_estimators=100,learning_rate=0.05,minibatch_frac=0.8,random_state=RANDOM_STATE,verbose=False),
 'decisiontree':DecisionTreeClassifier(random_state=RANDOM_STATE),
 'randomforest':RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE,n_jobs=-1),
 'xgboost':XGBClassifier(n_estimators=200,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,objective='multi:softprob',num_class=len(MULTICLASS_CLASSES),eval_metric='mlogloss',tree_method='hist',random_state=RANDOM_STATE,n_jobs=-1),
}
def multiclass_metrics(y_true,pred,proba,split):
    return {f'{split}_accuracy':accuracy_score(y_true,pred),f'{split}_balanced_accuracy':balanced_accuracy_score(y_true,pred),f'{split}_precision_macro':precision_score(y_true,pred,average='macro',zero_division=0),f'{split}_recall_macro':recall_score(y_true,pred,average='macro',zero_division=0),f'{split}_f1_macro':f1_score(y_true,pred,average='macro',zero_division=0),f'{split}_f1_micro':f1_score(y_true,pred,average='micro',zero_division=0),f'{split}_f1_weighted':f1_score(y_true,pred,average='weighted',zero_division=0),f'{split}_mcc':matthews_corrcoef(y_true,pred),f'{split}_log_loss':log_loss(y_true,proba,labels=np.arange(len(MULTICLASS_CLASSES))),f'{split}_roc_auc_ovr_macro':roc_auc_score(y_true,proba,multi_class='ovr',average='macro',labels=np.arange(len(MULTICLASS_CLASSES))),f'{split}_roc_auc_ovr_weighted':roc_auc_score(y_true,proba,multi_class='ovr',average='weighted',labels=np.arange(len(MULTICLASS_CLASSES)))}
train_mc_tracking=X_train_mc.copy(); train_mc_tracking['attack_category']=y_train_mc_text.to_numpy(); test_mc_tracking=X_test_mc.copy(); test_mc_tracking['attack_category']=y_test_mc_text.to_numpy()
train_mc_dataset=mlflow.data.from_pandas(train_mc_tracking,source=str(data_path_mc.resolve()),targets='attack_category',name='original_multiclass_train'); test_mc_dataset=mlflow.data.from_pandas(test_mc_tracking,source=str(data_path_mc.resolve()),targets='attack_category',name='original_multiclass_test')
def safe_param(value): return value if isinstance(value,(str,int,float,bool)) else str(value)[:500]

In [3]:
multiclass_results=[]; multiclass_per_class=[]
for algorithm_name,estimator in multiclass_models.items():
    run_name=f'original_Multiclass_{algorithm_name}'; pipeline_mc=Pipeline([('preprocessor',make_multiclass_preprocessor()),('model',estimator)])
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({'task':'multiclass_classification','data_variant':'Original','feature_set':'original','target':'attack_category','algorithm':algorithm_name,'class_weighting':'balanced_sample_weight'})
        mlflow.log_input(train_mc_dataset,context='training'); mlflow.log_input(test_mc_dataset,context='evaluation')
        mlflow.log_params({'algorithm':algorithm_name,'random_state':RANDOM_STATE,'target_column':'attack_category','class_count':len(MULTICLASS_CLASSES),'classes':','.join(MULTICLASS_CLASSES),'raw_feature_count':len(ORIGINAL_FEATURES_MC),'numeric_feature_count':len(numeric_mc),'categorical_feature_count':len(categorical_mc),'train_row_count':len(X_train_mc),'test_row_count':len(X_test_mc),'numeric_imputation':'median','numeric_scaling':'StandardScaler','categorical_imputation':'most_frequent','categorical_encoding':'OneHotEncoder(handle_unknown=ignore)','training_sample_weight':'sklearn_balanced_by_attack_category'})
        mlflow.log_params({f'model_{k}':safe_param(v) for k,v in estimator.get_params(deep=False).items() if v is not None})
        started=time.perf_counter(); pipeline_mc.fit(X_train_mc,y_train_mc,model__sample_weight=balanced_train_weights); fit_seconds=time.perf_counter()-started
        infer_started=time.perf_counter(); test_pred=pipeline_mc.predict(X_test_mc).astype(int); test_proba=pipeline_mc.predict_proba(X_test_mc); inference_seconds=time.perf_counter()-infer_started; train_pred=pipeline_mc.predict(X_train_mc).astype(int); train_proba=pipeline_mc.predict_proba(X_train_mc)
        metrics={**multiclass_metrics(y_train_mc,train_pred,train_proba,'train'),**multiclass_metrics(y_test_mc,test_pred,test_proba,'test'),'fit_seconds':fit_seconds,'test_inference_seconds':inference_seconds}
        per_precision,per_recall,per_f1,per_support=precision_recall_fscore_support(y_test_mc,test_pred,labels=np.arange(len(MULTICLASS_CLASSES)),zero_division=0); per_class_df=pd.DataFrame({'attack_category':MULTICLASS_CLASSES,'precision':per_precision,'recall_detection_rate':per_recall,'f1':per_f1,'support':per_support})
        for _,row in per_class_df.iterrows():
            slug=str(row.attack_category).lower(); true_binary=(y_test_mc==multiclass_mapping[row.attack_category]); pred_binary=(test_pred==multiclass_mapping[row.attack_category]); tn,fp,fn,tp=confusion_matrix(true_binary,pred_binary,labels=[False,True]).ravel(); metrics.update({f'test_{slug}_precision':float(row.precision),f'test_{slug}_recall':float(row.recall_detection_rate),f'test_{slug}_f1':float(row.f1),f'test_{slug}_false_positive_rate':fp/max(fp+tn,1)}); multiclass_per_class.append({'run_name':run_name,'run_id':run.info.run_id,'algorithm':algorithm_name,**row.to_dict(),'false_positive_rate':fp/max(fp+tn,1)})
        mlflow.log_metrics({k:float(v) for k,v in metrics.items()}); mlflow.log_table(per_class_df,'metrics/test_per_class_metrics.json'); mlflow.log_dict(classification_report(y_test_mc,test_pred,labels=np.arange(len(MULTICLASS_CLASSES)),target_names=MULTICLASS_CLASSES,output_dict=True,zero_division=0),'metrics/test_classification_report.json')
        mlflow.log_dict({'label_mapping':multiclass_mapping,'original_features':list(ORIGINAL_FEATURES_MC),'categorical_features':categorical_mc,'numeric_features':numeric_mc,'class_distribution_train':y_train_mc_text.value_counts().to_dict(),'class_distribution_test':y_test_mc_text.value_counts().to_dict(),'split_strategy':'same binary-stratified 80/20 split as original experiment','rare_class_warning':'U2R has very small support; interpret its metrics cautiously','library_versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__,'mlflow':mlflow.__version__,'ngboost':ngboost.__version__,'xgboost':xgboost.__version__}},'metadata/run_metadata.json')
        fig,axes=plt.subplots(1,2,figsize=(12,5)); ConfusionMatrixDisplay.from_predictions(y_test_mc,test_pred,display_labels=MULTICLASS_CLASSES,cmap='Blues',colorbar=False,ax=axes[0]); ConfusionMatrixDisplay.from_predictions(y_test_mc,test_pred,display_labels=MULTICLASS_CLASSES,normalize='true',values_format='.2f',cmap='Blues',colorbar=False,ax=axes[1]); axes[0].set_title('Counts'); axes[1].set_title('Normalized by true class'); fig.suptitle(run_name); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_confusion_matrices.png'); plt.close(fig)
        fig,ax=plt.subplots(figsize=(7,6)); y_binary=label_binarize(y_test_mc,classes=np.arange(len(MULTICLASS_CLASSES))); [RocCurveDisplay.from_predictions(y_binary[:,i],test_proba[:,i],name=class_name,ax=ax) for i,class_name in enumerate(MULTICLASS_CLASSES)]; ax.set_title(f'{run_name} - one-vs-rest ROC'); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_multiclass_roc.png'); plt.close(fig)
        fig,ax=plt.subplots(figsize=(8,4)); per_class_df.set_index('attack_category')[['precision','recall_detection_rate','f1']].plot(kind='bar',ax=ax); ax.set_ylim(0,1.05); ax.set_title(f'{run_name} - per-class metrics'); ax.tick_params(axis='x',rotation=30); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_per_class_metrics.png'); plt.close(fig)
        fitted_preprocessor=pipeline_mc.named_steps['preprocessor']; fitted_estimator=pipeline_mc.named_steps['model']; feature_names=np.asarray(fitted_preprocessor.get_feature_names_out()); importance=None
        if hasattr(fitted_estimator,'feature_importances_'): importance=np.asarray(fitted_estimator.feature_importances_); importance=np.mean(np.abs(importance),axis=0) if importance.ndim>1 else np.abs(importance)
        elif hasattr(fitted_estimator,'coef_'): importance=np.mean(np.abs(np.asarray(fitted_estimator.coef_)),axis=0)
        if importance is not None and len(importance)==len(feature_names):
            importance_df=pd.DataFrame({'feature':feature_names,'importance':importance}).sort_values('importance',ascending=False); mlflow.log_table(importance_df,'metrics/feature_importance.json'); fig,ax=plt.subplots(figsize=(8,6)); top=importance_df.head(25).sort_values('importance'); ax.barh(top.feature,top.importance); ax.set_title(f'{run_name} - top feature importance'); fig.tight_layout(); mlflow.log_figure(fig,'plots/feature_importance_top25.png'); plt.close(fig)
        input_example=X_train_mc.head(5).copy(); input_example[numeric_mc]=input_example[numeric_mc].astype('float64'); output_example=pd.DataFrame(test_proba[:5],columns=[f'probability_{c}' for c in MULTICLASS_CLASSES]); output_example.insert(0,'predicted_attack_category',[inverse_multiclass_mapping[int(v)] for v in test_pred[:5]]); mlflow.log_table(input_example.reset_index(drop=True),'examples/input_example.json'); mlflow.log_table(output_example,'examples/output_example.json'); mlflow.log_dict({'input':{'type':'pandas.DataFrame','required_columns':list(ORIGINAL_FEATURES_MC)},'predict_output':{'type':'integer class index','mapping':inverse_multiclass_mapping},'predict_proba_output':{'shape':['rows',len(MULTICLASS_CLASSES)],'column_order':MULTICLASS_CLASSES,'row_sum':1.0}},'metadata/input_output_contract.json')
        signature=infer_signature(input_example,pipeline_mc.predict(input_example)); mlflow.sklearn.log_model(sk_model=pipeline_mc,name='model',serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,signature=signature,input_example=input_example)
        multiclass_results.append({'run_name':run_name,'run_id':run.info.run_id,'algorithm':algorithm_name,**metrics}); print(f"{run_name}: macro-F1={metrics['test_f1_macro']:.4f}, weighted-F1={metrics['test_f1_weighted']:.4f}, balanced accuracy={metrics['test_balanced_accuracy']:.4f}")
multiclass_results_df=pd.DataFrame(multiclass_results).sort_values(['test_f1_macro','test_balanced_accuracy'],ascending=False).reset_index(drop=True); multiclass_per_class_df=pd.DataFrame(multiclass_per_class); display(multiclass_results_df[['run_name','test_accuracy','test_balanced_accuracy','test_precision_macro','test_recall_macro','test_f1_macro','test_f1_weighted','test_mcc','test_log_loss','test_roc_auc_ovr_macro']])

original_Multiclass_logisticreg: macro-F1=0.6816, weighted-F1=0.9733, balanced accuracy=0.8997


original_Multiclass_naivebayes: macro-F1=0.4078, weighted-F1=0.7511, balanced accuracy=0.5993


original_Multiclass_ngboost: macro-F1=0.7041, weighted-F1=0.9626, balanced accuracy=0.9339


original_Multiclass_decisiontree: macro-F1=0.9324, weighted-F1=0.9982, balanced accuracy=0.9171


original_Multiclass_randomforest: macro-F1=0.8358, weighted-F1=0.9989, balanced accuracy=0.8195


original_Multiclass_xgboost: macro-F1=0.9301, weighted-F1=0.9993, balanced accuracy=0.9723


,run_name,test_accuracy,test_balanced_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,test_mcc,test_log_loss,test_roc_auc_ovr_macro
0,original_Multiclass_decisiontree,0.998254,0.917052,0.953710,0.917052,0.932378,0.998246,0.996942,0.061551,0.958280
1,original_Multiclass_xgboost,0.999286,0.972345,0.903957,0.972345,0.930137,0.999319,0.998749,0.002932,0.999908
2,original_Multiclass_randomforest,0.998968,0.819484,0.897185,0.819484,0.835827,0.998872,0.998192,0.005794,0.987398
3,original_Multiclass_ngboost,0.957849,0.933909,0.664117,0.933909,0.704140,0.962593,0.927275,0.335050,0.994032
4,original_Multiclass_logisticreg,0.966422,0.899725,0.643444,0.899725,0.681614,0.973276,0.943967,0.111356,0.995926
5,original_Multiclass_naivebayes,0.687795,0.599301,0.553184,0.599301,0.407760,0.751072,0.547005,10.527464,0.899858


In [4]:
best_multiclass=multiclass_results_df.iloc[0]; best_class_rows=multiclass_per_class_df[multiclass_per_class_df.algorithm==best_multiclass.algorithm].sort_values('recall_detection_rate',ascending=False); easiest_attack=best_class_rows[best_class_rows.attack_category!='Normal'].iloc[0]; hardest_attack=best_class_rows[best_class_rows.attack_category!='Normal'].iloc[-1]
class_winners=multiclass_per_class_df[multiclass_per_class_df.attack_category!='Normal'].sort_values(['f1','recall_detection_rate'],ascending=False).groupby('attack_category',as_index=False).first()[['attack_category','algorithm','precision','recall_detection_rate','f1','false_positive_rate','support']]
display(Markdown('## Best algorithm for each attack category')); display(class_winners); display(Markdown('## Per-class results for the best overall model')); display(best_class_rows)
fig,ax=plt.subplots(figsize=(9,4)); comparison=multiclass_results_df.set_index('algorithm')[['test_f1_macro','test_f1_weighted','test_balanced_accuracy']]; comparison.plot(kind='bar',ax=ax); ax.set_ylim(0,1.05); ax.set_title('Original-feature multiclass model comparison'); ax.tick_params(axis='x',rotation=30); fig.tight_layout(); display(fig)
display(Markdown(f'''## Multiclass conclusion

The best overall original-feature multiclass model by macro-F1 is **{best_multiclass.algorithm}**. It achieves accuracy **{best_multiclass.test_accuracy:.4f}**, balanced accuracy **{best_multiclass.test_balanced_accuracy:.4f}**, macro-F1 **{best_multiclass.test_f1_macro:.4f}**, weighted-F1 **{best_multiclass.test_f1_weighted:.4f}**, MCC **{best_multiclass.test_mcc:.4f}**, and macro one-vs-rest ROC-AUC **{best_multiclass.test_roc_auc_ovr_macro:.4f}**.

For this model, the easiest non-normal family by recall is **{easiest_attack.attack_category}** ({easiest_attack.recall_detection_rate:.4f}) and the hardest is **{hardest_attack.attack_category}** ({hardest_attack.recall_detection_rate:.4f}). The `class_winners` table identifies the algorithm that specializes best in each attack family.

Accuracy and weighted-F1 are dominated by Normal and DoS traffic, so macro-F1, balanced accuracy, and the per-class tables should drive model selection. U2R has only **{int((y_test_mc_text=='U2R').sum())} test rows**, making its score highly uncertain; independent or temporal validation is required before deployment.'''))
with mlflow.start_run(run_id=str(best_multiclass.run_id)):
    mlflow.log_table(multiclass_results_df,'comparison/multiclass_model_comparison.json'); mlflow.log_table(class_winners,'comparison/best_algorithm_per_attack_category.json'); mlflow.log_figure(fig,'comparison/multiclass_summary.png')
plt.close(fig)

## Best algorithm for each attack category

,attack_category,algorithm,precision,recall_detection_rate,f1,false_positive_rate,support
0,DoS,randomforest,0.999676,0.999892,0.999784,0.000188,9259
1,Probe,xgboost,0.999563,0.999563,0.999563,0.000044,2289
2,R2L,xgboost,0.982456,0.988235,0.985337,0.000120,170
3,U2R,decisiontree,0.833333,0.625000,0.714286,0.000040,8


## Per-class results for the best overall model

,run_name,run_id,algorithm,attack_category,precision,recall_detection_rate,f1,support,false_positive_rate
16,original_Multiclass_decisiontree,b3089c3949a44112b6355e31e629c838,decisiontree,DoS,0.999460,0.999892,0.999676,9259,0.000314
15,original_Multiclass_decisiontree,b3089c3949a44112b6355e31e629c838,decisiontree,Normal,0.998885,0.997847,0.998366,13469,0.001279
17,original_Multiclass_decisiontree,b3089c3949a44112b6355e31e629c838,decisiontree,Probe,0.994340,0.997816,0.996075,2289,0.000568
18,original_Multiclass_decisiontree,b3089c3949a44112b6355e31e629c838,decisiontree,R2L,0.942529,0.964706,0.953488,170,0.000400
19,original_Multiclass_decisiontree,b3089c3949a44112b6355e31e629c838,decisiontree,U2R,0.833333,0.625000,0.714286,8,0.000040


<Figure size 900x400 with 1 Axes>

## Multiclass conclusion

The best overall original-feature multiclass model by macro-F1 is **decisiontree**. It achieves accuracy **0.9983**, balanced accuracy **0.9171**, macro-F1 **0.9324**, weighted-F1 **0.9982**, MCC **0.9969**, and macro one-vs-rest ROC-AUC **0.9583**.

For this model, the easiest non-normal family by recall is **DoS** (0.9999) and the hardest is **U2R** (0.6250). The `class_winners` table identifies the algorithm that specializes best in each attack family.

Accuracy and weighted-F1 are dominated by Normal and DoS traffic, so macro-F1, balanced accuracy, and the per-class tables should drive model selection. U2R has only **8 test rows**, making its score highly uncertain; independent or temporal validation is required before deployment.

# Stage 2 — Attack-family classification without Normal traffic

This experiment represents the second stage of a cascaded IDS. It uses only rows already known to be attacks and predicts one of four families: **DoS, Probe, R2L, or U2R**. The same six algorithms and 41 original features are evaluated. Runs follow `Original_attackclass_{algorithm}`.

In [3]:
ATTACK_ONLY_CLASSES=['DoS','Probe','R2L','U2R']; attack_only_mapping={name:i for i,name in enumerate(ATTACK_ONLY_CLASSES)}; inverse_attack_only_mapping={i:name for name,i in attack_only_mapping.items()}
attack_train_mask=y_train_mc_text.to_numpy()!='Normal'; attack_test_mask=y_test_mc_text.to_numpy()!='Normal'
X_train_attack=X_train_mc.iloc[np.flatnonzero(attack_train_mask)].copy(); X_test_attack=X_test_mc.iloc[np.flatnonzero(attack_test_mask)].copy(); y_train_attack_text=y_train_mc_text.iloc[np.flatnonzero(attack_train_mask)].reset_index(drop=True); y_test_attack_text=y_test_mc_text.iloc[np.flatnonzero(attack_test_mask)].reset_index(drop=True); y_train_attack=y_train_attack_text.map(attack_only_mapping).to_numpy(); y_test_attack=y_test_attack_text.map(attack_only_mapping).to_numpy(); attack_train_weights=compute_sample_weight(class_weight='balanced',y=y_train_attack)
class CompatibleAttackNGBClassifier(ClassifierMixin,NGBClassifier):
    def fit(self,X,y,**fit_params): fitted=super().fit(X,y,**fit_params); self.classes_=np.arange(len(ATTACK_ONLY_CLASSES)); return fitted
attack_only_models={
 'logisticreg':LogisticRegression(max_iter=1500,random_state=RANDOM_STATE),
 'naivebayes':GaussianNB(),
 'ngboost':CompatibleAttackNGBClassifier(Dist=k_categorical(len(ATTACK_ONLY_CLASSES)),n_estimators=100,learning_rate=0.05,minibatch_frac=0.8,random_state=RANDOM_STATE,verbose=False),
 'decisiontree':DecisionTreeClassifier(random_state=RANDOM_STATE),
 'randomforest':RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE,n_jobs=-1),
 'xgboost':XGBClassifier(n_estimators=200,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,objective='multi:softprob',num_class=len(ATTACK_ONLY_CLASSES),eval_metric='mlogloss',tree_method='hist',random_state=RANDOM_STATE,n_jobs=-1),
}
def attack_only_metrics(y_true,pred,proba,split):
    return {f'{split}_accuracy':accuracy_score(y_true,pred),f'{split}_balanced_accuracy':balanced_accuracy_score(y_true,pred),f'{split}_precision_macro':precision_score(y_true,pred,average='macro',zero_division=0),f'{split}_recall_macro':recall_score(y_true,pred,average='macro',zero_division=0),f'{split}_f1_macro':f1_score(y_true,pred,average='macro',zero_division=0),f'{split}_f1_micro':f1_score(y_true,pred,average='micro',zero_division=0),f'{split}_f1_weighted':f1_score(y_true,pred,average='weighted',zero_division=0),f'{split}_mcc':matthews_corrcoef(y_true,pred),f'{split}_log_loss':log_loss(y_true,proba,labels=np.arange(len(ATTACK_ONLY_CLASSES))),f'{split}_roc_auc_ovr_macro':roc_auc_score(y_true,proba,multi_class='ovr',average='macro',labels=np.arange(len(ATTACK_ONLY_CLASSES))),f'{split}_roc_auc_ovr_weighted':roc_auc_score(y_true,proba,multi_class='ovr',average='weighted',labels=np.arange(len(ATTACK_ONLY_CLASSES)))}
attack_train_tracking=X_train_attack.copy(); attack_train_tracking['attack_category']=y_train_attack_text.to_numpy(); attack_test_tracking=X_test_attack.copy(); attack_test_tracking['attack_category']=y_test_attack_text.to_numpy(); attack_train_dataset=mlflow.data.from_pandas(attack_train_tracking,source=str(data_path_mc.resolve()),targets='attack_category',name='original_attack_only_stage2_train'); attack_test_dataset=mlflow.data.from_pandas(attack_test_tracking,source=str(data_path_mc.resolve()),targets='attack_category',name='original_attack_only_stage2_test')
display(pd.DataFrame({'train':y_train_attack_text.value_counts(),'test':y_test_attack_text.value_counts()}).fillna(0).astype(int))

,train,test
attack_category,,
DoS,36668,9259
Probe,9367,2289
R2L,825,170
U2R,44,8


In [4]:
attack_only_results=[]; attack_only_per_class=[]
for algorithm_name,estimator in attack_only_models.items():
    run_name=f'Original_attackclass_{algorithm_name}'; pipeline_attack=Pipeline([('preprocessor',make_multiclass_preprocessor()),('model',estimator)])
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({'task':'attack_only_multiclass_classification','pipeline_stage':'stage_2','data_variant':'Original','feature_set':'original','target':'attack_category_without_normal','algorithm':algorithm_name,'class_weighting':'balanced_sample_weight'})
        mlflow.log_input(attack_train_dataset,context='attack_only_training'); mlflow.log_input(attack_test_dataset,context='attack_only_evaluation')
        mlflow.log_params({'algorithm':algorithm_name,'pipeline_stage':2,'random_state':RANDOM_STATE,'target_column':'attack_category','normal_excluded':True,'class_count':len(ATTACK_ONLY_CLASSES),'classes':','.join(ATTACK_ONLY_CLASSES),'raw_feature_count':len(ORIGINAL_FEATURES_MC),'numeric_feature_count':len(numeric_mc),'categorical_feature_count':len(categorical_mc),'train_row_count':len(X_train_attack),'test_row_count':len(X_test_attack),'numeric_imputation':'median','numeric_scaling':'StandardScaler','categorical_imputation':'most_frequent','categorical_encoding':'OneHotEncoder(handle_unknown=ignore)','training_sample_weight':'sklearn_balanced_by_attack_category'})
        mlflow.log_params({f'model_{k}':safe_param(v) for k,v in estimator.get_params(deep=False).items() if v is not None})
        started=time.perf_counter(); pipeline_attack.fit(X_train_attack,y_train_attack,model__sample_weight=attack_train_weights); fit_seconds=time.perf_counter()-started; infer_started=time.perf_counter(); test_pred=pipeline_attack.predict(X_test_attack).astype(int); test_proba=pipeline_attack.predict_proba(X_test_attack); inference_seconds=time.perf_counter()-infer_started; train_pred=pipeline_attack.predict(X_train_attack).astype(int); train_proba=pipeline_attack.predict_proba(X_train_attack)
        metrics={**attack_only_metrics(y_train_attack,train_pred,train_proba,'train'),**attack_only_metrics(y_test_attack,test_pred,test_proba,'test'),'fit_seconds':fit_seconds,'test_inference_seconds':inference_seconds}; per_precision,per_recall,per_f1,per_support=precision_recall_fscore_support(y_test_attack,test_pred,labels=np.arange(len(ATTACK_ONLY_CLASSES)),zero_division=0); per_class_df=pd.DataFrame({'attack_category':ATTACK_ONLY_CLASSES,'precision':per_precision,'recall_detection_rate':per_recall,'f1':per_f1,'support':per_support})
        for _,row in per_class_df.iterrows():
            slug=str(row.attack_category).lower(); true_binary=(y_test_attack==attack_only_mapping[row.attack_category]); pred_binary=(test_pred==attack_only_mapping[row.attack_category]); tn,fp,fn,tp=confusion_matrix(true_binary,pred_binary,labels=[False,True]).ravel(); fpr=fp/max(fp+tn,1); metrics.update({f'test_{slug}_precision':float(row.precision),f'test_{slug}_recall':float(row.recall_detection_rate),f'test_{slug}_f1':float(row.f1),f'test_{slug}_false_positive_rate':fpr}); attack_only_per_class.append({'run_name':run_name,'run_id':run.info.run_id,'algorithm':algorithm_name,**row.to_dict(),'false_positive_rate':fpr})
        mlflow.log_metrics({k:float(v) for k,v in metrics.items()}); mlflow.log_table(per_class_df,'metrics/test_per_class_metrics.json'); mlflow.log_dict(classification_report(y_test_attack,test_pred,labels=np.arange(len(ATTACK_ONLY_CLASSES)),target_names=ATTACK_ONLY_CLASSES,output_dict=True,zero_division=0),'metrics/test_classification_report.json')
        mlflow.log_dict({'pipeline_stage':2,'normal_rows_excluded':True,'label_mapping':attack_only_mapping,'original_features':list(ORIGINAL_FEATURES_MC),'categorical_features':categorical_mc,'numeric_features':numeric_mc,'class_distribution_train':y_train_attack_text.value_counts().to_dict(),'class_distribution_test':y_test_attack_text.value_counts().to_dict(),'upstream_contract':'Stage 1 must route only predicted attacks to this model','split_strategy':'same original binary-stratified 80/20 split, filtered to attack rows','rare_class_warning':'U2R support is extremely small','library_versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__,'mlflow':mlflow.__version__,'ngboost':ngboost.__version__,'xgboost':xgboost.__version__}},'metadata/run_metadata.json')
        fig,axes=plt.subplots(1,2,figsize=(12,5)); ConfusionMatrixDisplay.from_predictions(y_test_attack,test_pred,display_labels=ATTACK_ONLY_CLASSES,cmap='Blues',colorbar=False,ax=axes[0]); ConfusionMatrixDisplay.from_predictions(y_test_attack,test_pred,display_labels=ATTACK_ONLY_CLASSES,normalize='true',values_format='.2f',cmap='Blues',colorbar=False,ax=axes[1]); axes[0].set_title('Counts'); axes[1].set_title('Normalized by true class'); fig.suptitle(run_name); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_confusion_matrices.png'); plt.close(fig)
        fig,ax=plt.subplots(figsize=(7,6)); y_binary=label_binarize(y_test_attack,classes=np.arange(len(ATTACK_ONLY_CLASSES))); [RocCurveDisplay.from_predictions(y_binary[:,i],test_proba[:,i],name=class_name,ax=ax) for i,class_name in enumerate(ATTACK_ONLY_CLASSES)]; ax.set_title(f'{run_name} - one-vs-rest ROC'); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_attack_family_roc.png'); plt.close(fig)
        fig,ax=plt.subplots(figsize=(8,4)); per_class_df.set_index('attack_category')[['precision','recall_detection_rate','f1']].plot(kind='bar',ax=ax); ax.set_ylim(0,1.05); ax.set_title(f'{run_name} - per-family metrics'); ax.tick_params(axis='x',rotation=30); fig.tight_layout(); mlflow.log_figure(fig,'plots/test_per_family_metrics.png'); plt.close(fig)
        fitted_preprocessor=pipeline_attack.named_steps['preprocessor']; fitted_estimator=pipeline_attack.named_steps['model']; feature_names=np.asarray(fitted_preprocessor.get_feature_names_out()); importance=None
        if hasattr(fitted_estimator,'feature_importances_'): importance=np.asarray(fitted_estimator.feature_importances_); importance=np.mean(np.abs(importance),axis=0) if importance.ndim>1 else np.abs(importance)
        elif hasattr(fitted_estimator,'coef_'): importance=np.mean(np.abs(np.asarray(fitted_estimator.coef_)),axis=0)
        if importance is not None and len(importance)==len(feature_names):
            importance_df=pd.DataFrame({'feature':feature_names,'importance':importance}).sort_values('importance',ascending=False); mlflow.log_table(importance_df,'metrics/feature_importance.json'); fig,ax=plt.subplots(figsize=(8,6)); top=importance_df.head(25).sort_values('importance'); ax.barh(top.feature,top.importance); ax.set_title(f'{run_name} - top feature importance'); fig.tight_layout(); mlflow.log_figure(fig,'plots/feature_importance_top25.png'); plt.close(fig)
        input_example=X_train_attack.head(5).copy(); input_example[numeric_mc]=input_example[numeric_mc].astype('float64'); output_example=pd.DataFrame(test_proba[:5],columns=[f'probability_{c}' for c in ATTACK_ONLY_CLASSES]); output_example.insert(0,'predicted_attack_category',[inverse_attack_only_mapping[int(v)] for v in test_pred[:5]]); mlflow.log_table(input_example.reset_index(drop=True),'examples/input_example.json'); mlflow.log_table(output_example,'examples/output_example.json'); mlflow.log_dict({'stage':2,'input':{'type':'pandas.DataFrame','required_columns':list(ORIGINAL_FEATURES_MC),'precondition':'Stage 1 classified row as Attack'},'predict_output':{'type':'integer class index','mapping':inverse_attack_only_mapping},'predict_proba_output':{'shape':['rows',len(ATTACK_ONLY_CLASSES)],'column_order':ATTACK_ONLY_CLASSES,'row_sum':1.0}},'metadata/input_output_contract.json')
        signature=infer_signature(input_example,pipeline_attack.predict(input_example)); mlflow.sklearn.log_model(sk_model=pipeline_attack,name='stage2_attack_family_model',serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,signature=signature,input_example=input_example)
        attack_only_results.append({'run_name':run_name,'run_id':run.info.run_id,'algorithm':algorithm_name,**metrics}); print(f"{run_name}: macro-F1={metrics['test_f1_macro']:.4f}, balanced accuracy={metrics['test_balanced_accuracy']:.4f}, weighted-F1={metrics['test_f1_weighted']:.4f}")
attack_only_results_df=pd.DataFrame(attack_only_results).sort_values(['test_f1_macro','test_balanced_accuracy'],ascending=False).reset_index(drop=True); attack_only_per_class_df=pd.DataFrame(attack_only_per_class); display(attack_only_results_df[['run_name','test_accuracy','test_balanced_accuracy','test_precision_macro','test_recall_macro','test_f1_macro','test_f1_weighted','test_mcc','test_log_loss','test_roc_auc_ovr_macro']])

Original_attackclass_logisticreg: macro-F1=0.8535, balanced accuracy=0.9274, weighted-F1=0.9983


Original_attackclass_naivebayes: macro-F1=0.5601, balanced accuracy=0.7540, weighted-F1=0.9415


Original_attackclass_ngboost: macro-F1=0.8709, balanced accuracy=0.9245, weighted-F1=0.9843


Original_attackclass_decisiontree: macro-F1=0.9143, balanced accuracy=0.9033, weighted-F1=0.9996


Original_attackclass_randomforest: macro-F1=0.9832, balanced accuracy=0.9687, weighted-F1=0.9998


Original_attackclass_xgboost: macro-F1=0.9678, balanced accuracy=0.9672, weighted-F1=0.9997


,run_name,test_accuracy,test_balanced_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,test_mcc,test_log_loss,test_roc_auc_ovr_macro
0,Original_attackclass_randomforest,0.999829,0.968723,0.999782,0.968723,0.983211,0.999827,0.999496,0.001984,0.999971
1,Original_attackclass_xgboost,0.999659,0.967225,0.968423,0.967225,0.967822,0.999659,0.998992,0.000943,0.999992
2,Original_attackclass_decisiontree,0.999574,0.903309,0.926874,0.903309,0.914345,0.999559,0.998739,0.015369,0.951596
3,Original_attackclass_ngboost,0.983370,0.924469,0.838076,0.924469,0.870860,0.984322,0.951819,0.067440,0.996278
4,Original_attackclass_logisticreg,0.998039,0.927367,0.823855,0.927367,0.853479,0.998295,0.994206,0.013002,0.997170
5,Original_attackclass_naivebayes,0.932799,0.753990,0.555852,0.753990,0.560117,0.941540,0.802923,2.274830,0.939629


In [5]:
best_attack_only=attack_only_results_df.iloc[0]; best_attack_rows=attack_only_per_class_df[attack_only_per_class_df.algorithm==best_attack_only.algorithm].sort_values('recall_detection_rate',ascending=False); family_winners=attack_only_per_class_df.sort_values(['f1','recall_detection_rate'],ascending=False).groupby('attack_category',as_index=False).first()[['attack_category','algorithm','precision','recall_detection_rate','f1','false_positive_rate','support']]
display(Markdown('## Best Stage-2 algorithm for each attack family')); display(family_winners); display(Markdown('## Per-family performance of the best overall Stage-2 model')); display(best_attack_rows)
fig,ax=plt.subplots(figsize=(9,4)); attack_only_results_df.set_index('algorithm')[['test_f1_macro','test_f1_weighted','test_balanced_accuracy']].plot(kind='bar',ax=ax); ax.set_ylim(0,1.05); ax.set_title('Attack-only Stage-2 model comparison'); ax.tick_params(axis='x',rotation=30); fig.tight_layout(); display(fig)
experiment_mc=mlflow.get_experiment_by_name(EXPERIMENT_NAME); previous_five_class=mlflow.search_runs([experiment_mc.experiment_id],filter_string="tags.mlflow.runName = 'original_Multiclass_xgboost'",order_by=['start_time DESC'],max_results=1); five_class_macro=float(previous_five_class.iloc[0]['metrics.test_f1_macro']) if len(previous_five_class) else np.nan
display(Markdown(f'''# Stage-2 attack-only conclusions

The best attack-only model by macro-F1 is **{best_attack_only.algorithm}** with accuracy **{best_attack_only.test_accuracy:.4f}**, balanced accuracy **{best_attack_only.test_balanced_accuracy:.4f}**, macro-F1 **{best_attack_only.test_f1_macro:.4f}**, weighted-F1 **{best_attack_only.test_f1_weighted:.4f}**, MCC **{best_attack_only.test_mcc:.4f}**, log loss **{best_attack_only.test_log_loss:.4f}**, and macro ROC-AUC **{best_attack_only.test_roc_auc_ovr_macro:.4f}**.

Removing Normal aligns training with the real Stage-2 input contract and prevents the dominant Normal class from influencing family boundaries. The previous five-class XGBoost macro-F1 was **{five_class_macro:.4f}**; compare it with the attack-only results using macro-F1 and per-family recall rather than overall accuracy.

The `family_winners` table shows whether one algorithm dominates all families or whether rare R2L/U2R traffic benefits from a specialist. U2R has only **{int((y_test_attack_text=='U2R').sum())} test samples**, so its apparent winner is unstable. Production selection requires repeated stratified or temporal validation and probability calibration.

For the complete cascade, end-to-end recall for family `k` is approximately `Stage-1 attack recall × Stage-2 recall(k)`. Therefore Stage 1 should be thresholded for very high attack recall, while Stage 2 should be selected primarily by macro-F1, balanced accuracy, per-family recall, and calibrated probabilities.'''))
with mlflow.start_run(run_id=str(best_attack_only.run_id)):
    mlflow.log_table(attack_only_results_df,'comparison/stage2_attack_only_model_comparison.json'); mlflow.log_table(family_winners,'comparison/best_stage2_algorithm_per_family.json'); mlflow.log_figure(fig,'comparison/stage2_attack_only_summary.png')
plt.close(fig)

## Best Stage-2 algorithm for each attack family

,attack_category,algorithm,precision,recall_detection_rate,f1,false_positive_rate,support
0,DoS,decisiontree,1.000000,1.000,1.000000,0.000000,9259
1,Probe,decisiontree,0.999127,1.000,0.999563,0.000212,2289
2,R2L,randomforest,1.000000,1.000,1.000000,0.000000,170
3,U2R,randomforest,1.000000,0.875,0.933333,0.000000,8


## Per-family performance of the best overall Stage-2 model

,run_name,run_id,algorithm,attack_category,precision,recall_detection_rate,f1,support,false_positive_rate
17,Original_attackclass_randomforest,2e05768f7e1444b2917118337a8ccf95,randomforest,Probe,0.999127,1.000000,0.999563,2289,0.000212
18,Original_attackclass_randomforest,2e05768f7e1444b2917118337a8ccf95,randomforest,R2L,1.000000,1.000000,1.000000,170,0.000000
16,Original_attackclass_randomforest,2e05768f7e1444b2917118337a8ccf95,randomforest,DoS,1.000000,0.999892,0.999946,9259,0.000000
19,Original_attackclass_randomforest,2e05768f7e1444b2917118337a8ccf95,randomforest,U2R,1.000000,0.875000,0.933333,8,0.000000


<Figure size 900x400 with 1 Axes>

# Stage-2 attack-only conclusions

The best attack-only model by macro-F1 is **randomforest** with accuracy **0.9998**, balanced accuracy **0.9687**, macro-F1 **0.9832**, weighted-F1 **0.9998**, MCC **0.9995**, log loss **0.0020**, and macro ROC-AUC **1.0000**.

Removing Normal aligns training with the real Stage-2 input contract and prevents the dominant Normal class from influencing family boundaries. The previous five-class XGBoost macro-F1 was **0.9301**; compare it with the attack-only results using macro-F1 and per-family recall rather than overall accuracy.

The `family_winners` table shows whether one algorithm dominates all families or whether rare R2L/U2R traffic benefits from a specialist. U2R has only **8 test samples**, so its apparent winner is unstable. Production selection requires repeated stratified or temporal validation and probability calibration.

For the complete cascade, end-to-end recall for family `k` is approximately `Stage-1 attack recall × Stage-2 recall(k)`. Therefore Stage 1 should be thresholded for very high attack recall, while Stage 2 should be selected primarily by macro-F1, balanced accuracy, per-family recall, and calibrated probabilities.